# Session 2 Gates & the Bloch Sphere · Quantum Computing Workshop

**The rituals still rule:**
- Run cells **top to bottom**.  **YOUR TURN** = fill in a blank.  **PREDICT FIRST** = type your prediction *before* running.
- Pairs: driver types, navigator reads aloud. **Swap at Task 3.**

> First: **File → Save a copy in Drive**.

## Cell 1 — Setup (run first, ~1 minute)

In [ ]:
%pip install -q qiskit qiskit-aer matplotlib pylatexenc

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector, Operator
from qiskit.visualization import plot_histogram, plot_bloch_multivector
import numpy as np

simulator = AerSimulator()

def show_ball(qc):
    """Draw the Bloch sphere for a (measurement-free) 1-qubit circuit."""
    return plot_bloch_multivector(Statevector(qc))

def show_recipe(qc):
    """Print the state's amplitudes: the recipe a·|0⟩ + b·|1⟩."""
    a, b = np.round(Statevector(qc).data, 3)
    print(f"amplitude of |0⟩:  {a}")
    print(f"amplitude of |1⟩:  {b}")

def run_counts(qc, shots=1000):
    return simulator.run(qc, shots=shots).result().get_counts()

print("✅ Setup complete")

---
## Task 1 — See the sphere in code

`show_ball(qc)` draws the Bloch sphere with an arrow showing exactly where your qubit is. **No measurement in these circuits** — we're looking at the state itself, a luxury only simulators have (peeking at a *real* qubit collapses it).

Start at the north pole:

In [ ]:
qc = QuantumCircuit(1)   # fresh qubit = |0⟩ = north pole
show_ball(qc)

** PREDICT FIRST:** where does the arrow point after `x`? After `h`? After `h` then `s`? Sketch or say it, *then* run each cell. (Your sticky-note ball from class is legal equipment.)

In [ ]:
qc = QuantumCircuit(1)
qc.x(0)
show_ball(qc)   # prediction: ______

In [ ]:
qc = QuantumCircuit(1)
qc.h(0)
show_ball(qc)   # prediction: ______

In [ ]:
qc = QuantumCircuit(1)
qc.h(0)
qc.s(0)
show_ball(qc)   # prediction: ______

<details><summary> Check yourself</summary>

`x` → south pole |1⟩ (half-turn about the east–west skewer). `h` → east |+⟩ (the pole↔ring bridge). `h` then `s` → the ring spot one stop past east: |+i⟩ (S = quarter-twist of longitude, poles untouched).
</details>

---
## Task 2 — Read the Recipe

The ball is geometry; the **recipe is the numbers**: every state is `a·|0⟩ + b·|1⟩`, and `show_recipe(qc)` prints `a` and `b` straight from the simulator.

** PREDICT FIRST:** a fresh qubit hit with `h` becomes the spinning coin |+⟩. What two numbers should print? (Hint from class: 1/√2 ≈ 0.707.)

In [ ]:
qc = QuantumCircuit(1)
qc.h(0)
show_recipe(qc)   # prediction: a = ____ , b = ____

**PREDICT FIRST, round two:** now `x` *then* `h` — Recipe A from the case file. In class we claimed this lands on the **evil twin** |−⟩. If that's true, what should print?

In [ ]:
qc = QuantumCircuit(1)
qc.x(0)
qc.h(0)
show_recipe(qc)   # prediction: a = ____ , b = ____

**There it is: a minus sign.** The invisible thing from Session 1 is just… a negative number in the recipe.

Now watch measurement eliminate it. Squaring converts amplitudes → probabilities (we use `abs()**2` so it also handles the `i`-cousins):

In [ ]:
amps = Statevector(qc).data
probs = np.abs(amps)**2
print("P(0) =", round(probs[0], 3), "   P(1) =", round(probs[1], 3))

 **YOUR TURN:** build the circuit `h` then `s` and call `show_recipe`. **PREDICT FIRST:** which cousin is this, and what will the |1⟩-amplitude look like? Then compute its probabilities — still 50/50?

In [ ]:
# your circuit here
qc = QuantumCircuit(1)
qc.h(0)
qc.s(0)
show_recipe(qc) 

<details><summary> Solution</summary>

```python
qc = QuantumCircuit(1)
qc.h(0)
qc.s(0)
show_recipe(qc)        # amplitude of |1⟩: 0.707j  ← that's Python for 0.707·i
amps = Statevector(qc).data
print(np.abs(amps)**2) # [0.5, 0.5]
```
It's |+i⟩ — the quarter-turn cousin. `abs` squares away the `i` just like it squares away a minus sign: **all four ring recipes measure 50/50.** Four different numbers, one histogram — which is exactly why we needed the ball.
</details>

---
## Task 3 — Close the case file
*(Drivers and navigators: swap seats!)*

Session 1, Mission E2: two recipes, identical 50/50 histograms, yet provably different. You now have **two** instruments the histogram lacks. Use both.

In [ ]:
recipeA = QuantumCircuit(1)
recipeA.x(0)
recipeA.h(0)
show_recipe(recipeA)     # the numbers…
show_ball(recipeA)       # …and the map. Recipe A: x then h

In [ ]:
recipeB = QuantumCircuit(1)
recipeB.h(0)
recipeB.x(0)
show_recipe(recipeB)
show_ball(recipeB)       # Recipe B: h then x

**Same latitude (equator → both 50/50), opposite longitude (west |−⟩ vs east |+⟩) — and the recipes differ by exactly one minus sign.**

 **YOUR TURN:** append `recipeA.h(0)` / `recipeB.h(0)` in each cell above (before the `show_` lines) and re-run. **PREDICT FIRST:** what will the recipes become? Where do the arrows swing?

<details><summary> What you should see</summary>

Recipe A: `[0, 1]` — the south pole, 100% ones. Recipe B: `[1, 0]` — north pole, 100% zeros. The final `h` made the |0⟩-parts (A) or |1⟩-parts (B) **cancel** — the whiteboard arithmetic from class, performed by the machine. Hidden sign → visible odds: **interference**, the engine of Session 4.
</details>

---
## Task 4 — The latitude dial: `ry`

The board game jumps between 6 landmarks. `qc.ry(angle, 0)` **glides**: it tilts the arrow `angle` radians south from the north pole. The recipe follows one formula:

$$P(\text{measure } 0) = \cos^2(\text{angle}/2)$$

Sanity-check the edge cases first:

In [ ]:
for angle in [0, np.pi/2, np.pi]:
    qc = QuantumCircuit(1, 1)
    qc.ry(angle, 0)
    qc.measure(0, 0)
    print(f"angle = {angle:.3f} rad  →", run_counts(qc))

 **YOUR TURN — the exact 75/25 coin.** In Session 1 you hunted a biased coin by trial and error. You've since acquired a formula. **Derive** the angle with cos²(angle/2) = 0.75, plug it in, and let 1000 shots judge you.

*No trig in your toolkit? Use the inverse recipe: `angle = 2 * np.arccos(np.sqrt(0.75))` — but first guess which "nice" angle it'll turn out to be.*

In [ ]:
my_angle = np.pi/3    # ← your derived angle (radians!)

qc = QuantumCircuit(1, 1)
qc.ry(my_angle, 0)
qc.measure(0, 0)
counts = run_counts(qc)
print(counts)
plot_histogram(counts)

<details><summary> Solution</summary>

cos²(θ/2) = 3/4 → cos(θ/2) = √3/2 → θ/2 = π/6 → **θ = π/3** (60° — told you it was beautiful). `my_angle = np.pi/3`. Expect ~750/250 with statistical wobble.

Bonus check with your Mission 2 skills: build the circuit *without* measurement and `show_recipe` it — you should see ≈ `[0.866, 0.5]`, and 0.866² = 0.75. Formula, recipe, histogram: all telling one story.
</details>

---
## Task 5 — Prove your golf round 

Two circuits are *the same gate* if they're the same rotation of the ball. One line checks it — global phase (an overall sign on the whole recipe, the invisible kind) is politely ignored:

In [ ]:
def same_gate(circuit1, circuit2):
    return Operator(circuit1).equiv(Operator(circuit2))

# Hole 1 verification: does H,Z,H really equal X?
hole1 = QuantumCircuit(1); hole1.h(0); hole1.z(0); hole1.h(0)
target = QuantumCircuit(1); target.x(0)
print("HZH == X :", same_gate(hole1, target))

**YOUR TURN:** machine-verify the rest of your scorecard — Hole 2 (HXH vs Z), Hole 3 (TT vs S), Hole 4 (your 4-gate identity vs an empty circuit), Hole 5 (HZHZ vs Y). One cell per hole, or a loop if you're feeling fancy.

In [ ]:
# your verifications here


<details><summary> Solutions</summary>

```python
h2 = QuantumCircuit(1); h2.h(0); h2.x(0); h2.h(0)
z  = QuantumCircuit(1); z.z(0)
print(same_gate(h2, z))          # True

h3 = QuantumCircuit(1); h3.t(0); h3.t(0)
s_ = QuantumCircuit(1); s_.s(0)
print(same_gate(h3, s_))         # True

h4 = QuantumCircuit(1); h4.h(0); h4.z(0); h4.h(0); h4.x(0)
nothing = QuantumCircuit(1)
print(same_gate(h4, idnetity))    # True  (HZH=X, then X·X = identity)

h5 = QuantumCircuit(1); h5.h(0); h5.z(0); h5.h(0); h5.z(0)
y  = QuantumCircuit(1); y.y(0)
print(same_gate(h5, y))          # True  (up to invisible global phase)
```
Fun trap: try `h,x,h,x` on Hole 4 — the verifier says it's NOT "nothing" (it's secretly Y). The machine keeps golf honest.
</details>

---
---
# Explore (optional, any order — predict before every run!)

### E1 — The ghost dial
`rz(angle)` rotates **longitude only** — pure phase, any amount you like. Prove it's invisible: measure after `h` + `rz(anything)`. Then try a **H sandwich** and measure again.

In [ ]:
ghost = QuantumCircuit(1, 1)
ghost.h(0)
ghost.rz(np.pi/3, 0)     # spin the hidden dial by 60°
ghost.measure(0, 0)
print("ghost alone:   ", run_counts(ghost))

trapped = QuantumCircuit(1, 1)
trapped.h(0)
trapped.rz(np.pi/3, 0)
trapped.h(0)             # the sandwich lid
trapped.measure(0, 0)
print("in H sandwich: ", run_counts(trapped))

<details><summary> What just happened</summary>

Alone: ~50/50 no matter the angle — phase is measurement-proof. Sandwiched: **~75/25**, following cos²(angle/2) — the SAME formula as `ry`! The H sandwich converted a longitude dial into a latitude dial. You just performed interference with a dial in your hand; every Session 4 algorithm is this trick, choreographed.
</details>

### E2 — The quarter-turn
`t` is the eighth-twist. Predict, then verify with `same_gate`: how many `t`'s make an `s`? A `z`? A "do nothing"/identity?

In [ ]:
# build and check  here


<details><summary> Solution</summary>

2 t's = S, 4 t's = Z, 8 t's = nothing (a full 360° of longitude). The T→S→Z ladder of ever-finer dials is real hardware's daily bread.
</details>

### E3 — The 99/1 coin
Design a coin that lands 0 exactly 99% of the time. Derive or use the inverse recipe — then check with 10,000 shots (why more shots for this one?).

In [ ]:
# your 99/1 coin here


<details><summary> Solution</summary>

`angle = 2*np.arccos(np.sqrt(0.99))` ≈ 0.2003 rad. 10,000 shots because at 1,000 you'd expect only ~10 ones — too few to eyeball 1% vs 2%. (Statistics matters: this is exactly how hardware calibration teams think.)
</details>

### E4 — When does phase do NOTHING?
`z` on |0⟩, then `z` on |+⟩ (i.e. after an `h`). Use `show_recipe` AND `show_ball` on both. One is changed, one isn't — explain with either instrument.

In [ ]:
# investigate here


<details><summary> Solution</summary>

Recipe view: `z` multiplies the |1⟩-part by −1. |0⟩ has **no** |1⟩-part — nothing to flip (and a sign on a lone term is global = invisible anyway). |+⟩ becomes |−⟩ — a *relative* sign between two parts: real. Ball view: the north pole sits ON the Z spin axis; |+⟩ sits on the equator and swings west. Same verdict, two instruments. Moral: phase only *exists* relative to a superposition.
</details>

### E5 — For more experienced coders 
Write `make_coin(p)` returning a 1-qubit circuit (with measurement) that outputs 0 with probability `p`. Test it across `p in [0.1, 0.3, 0.5, 0.9]` in a loop, 10,000 shots each, and print measured-vs-target.

In [ ]:
def make_coin(p):
    qc = QuantumCircuit(1, 1)
  # your code here
    return qc

# test code here

<details><summary>Solution</summary>

```python
def make_coin(p):
    qc = QuantumCircuit(1, 1)
    qc.ry(2*np.arccos(np.sqrt(p)), 0)
    qc.measure(0, 0)
    return qc

for p in [0.1, 0.3, 0.5, 0.9]:
    c = run_counts(make_coin(p), shots=10000)
    print(f"target {p:.0%} → measured {c.get('0',0)/10000:.1%}")
```
Congratulations: you've written a tiny quantum library. Session 3's Bell states are two-qubit — one more wire and things get *spooky*.
</details>

---
##  Done!
You can now: **write** a quantum state, read its amplitudes off the screen, dial any latitude exactly, verify gate identities like a compiler engineer, and — twice today, once by hand and once in E1 — you performed **interference**. Next session: two qubits, one fate, and hopefully real hardware.